In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import missingno as msno
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold
import numpy as np
from sklearn.base import clone
import importlib
import utils
importlib.reload(utils)
from utils import plot_y_yhat,clean_train_data,error_metric, predict_average,baseline_cross_validation
from sklearn.experimental import enable_iterative_imputer  
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.impute import MissingIndicator


In [44]:
train=pd.read_csv('../data/train_data.csv')
test=pd.read_csv('../data/test_data.csv')
sample_submission=pd.read_csv('../data/sample_submission.csv')

In [45]:
train.head()

,id,Age,Gender,Stage,GeneticRisk,TreatmentType,ComorbidityIndex,TreatmentResponse,SurvivalTime,Censored
0,0,65.0,0,2,1.0,0,0.0,0.0,4.2,0
1,1,68.0,1,2,0.0,1,1.0,1.0,4.7,1
2,2,68.0,1,3,1.0,1,0.0,1.0,3.5,1
3,3,81.0,1,4,1.0,1,3.0,0.0,2.3,0
4,4,59.0,1,2,1.0,0,NaN,0.0,NaN,0


In [46]:
train.isna().sum()

id                     0
Age                    0
Gender                 0
Stage                  0
GeneticRisk           85
TreatmentType          0
ComorbidityIndex      45
TreatmentResponse     29
SurvivalTime         160
Censored               0
dtype: int64

In [47]:
cols_to_impute = ["GeneticRisk", "ComorbidityIndex", "TreatmentResponse"]
df = train.copy()

In [48]:
# --- Zero Imputation ---
imputer = SimpleImputer(strategy="constant", fill_value=0, add_indicator=True)
transformed = imputer.fit_transform(df[cols_to_impute])

# Indicator columns for missing values
indicator_cols = [f"{col}_missing" for col in cols_to_impute]

df_zero = df.copy()
df_zero[cols_to_impute + indicator_cols] = transformed

print("Zero Imputation - Remaining NaNs:\n", df_zero.isna().sum())


Zero Imputation - Remaining NaNs:
 id                             0
Age                            0
Gender                         0
Stage                          0
GeneticRisk                    0
TreatmentType                  0
ComorbidityIndex               0
TreatmentResponse              0
SurvivalTime                 160
Censored                       0
GeneticRisk_missing            0
ComorbidityIndex_missing       0
TreatmentResponse_missing      0
dtype: int64


In [49]:
# --- Mean Imputation ---
imputer = SimpleImputer(strategy="mean", add_indicator=True)
transformed = imputer.fit_transform(df[cols_to_impute])

df_mean = df.copy()
df_mean[cols_to_impute + indicator_cols] = transformed

print("Mean Imputation - Remaining NaNs:\n", df_mean.isna().sum())

Mean Imputation - Remaining NaNs:
 id                             0
Age                            0
Gender                         0
Stage                          0
GeneticRisk                    0
TreatmentType                  0
ComorbidityIndex               0
TreatmentResponse              0
SurvivalTime                 160
Censored                       0
GeneticRisk_missing            0
ComorbidityIndex_missing       0
TreatmentResponse_missing      0
dtype: int64


In [50]:
# --- Median Imputation ---
imputer = SimpleImputer(strategy="median", add_indicator=True)
transformed = imputer.fit_transform(df[cols_to_impute])

df_median = df.copy()
df_median[cols_to_impute + indicator_cols] = transformed

print("Median Imputation - Remaining NaNs:\n", df_median.isna().sum())

Median Imputation - Remaining NaNs:
 id                             0
Age                            0
Gender                         0
Stage                          0
GeneticRisk                    0
TreatmentType                  0
ComorbidityIndex               0
TreatmentResponse              0
SurvivalTime                 160
Censored                       0
GeneticRisk_missing            0
ComorbidityIndex_missing       0
TreatmentResponse_missing      0
dtype: int64


In [51]:
# --- KNN Imputation ---
imputer = KNNImputer(n_neighbors=5, weights="uniform")
transformed = imputer.fit_transform(df[cols_to_impute])

df_knn = df.copy()
df_knn[cols_to_impute] = transformed  # KNN does not add indicator columns

print("KNN Imputation - Remaining NaNs:\n", df_knn.isna().sum())

KNN Imputation - Remaining NaNs:
 id                     0
Age                    0
Gender                 0
Stage                  0
GeneticRisk            0
TreatmentType          0
ComorbidityIndex       0
TreatmentResponse      0
SurvivalTime         160
Censored               0
dtype: int64


In [52]:
# --- Iterative Imputation ---
imputer = IterativeImputer(max_iter=10, random_state=42)
transformed = imputer.fit_transform(df[cols_to_impute])

df_iter = df.copy()
df_iter[cols_to_impute] = transformed  # Iterative does not add indicator columns

print("Iterative Imputation - Remaining NaNs:\n", df_iter.isna().sum())

Iterative Imputation - Remaining NaNs:
 id                     0
Age                    0
Gender                 0
Stage                  0
GeneticRisk            0
TreatmentType          0
ComorbidityIndex       0
TreatmentResponse      0
SurvivalTime         160
Censored               0
dtype: int64


In [53]:
# --- Most Frequent Imputation ---
# Goal: Replace missing values using the most frequent value of each column

imputer = SimpleImputer(strategy="most_frequent", add_indicator=True)
transformed = imputer.fit_transform(df[cols_to_impute])

# Create new indicator columns (same naming as your previous blocks)
indicator_cols = [f"{col}_missing" for col in cols_to_impute]

df_mostfreq = df.copy()
df_mostfreq[cols_to_impute + indicator_cols] = transformed

print("Most Frequent Imputation - Remaining NaNs:\n", df_mostfreq.isna().sum())

Most Frequent Imputation - Remaining NaNs:
 id                             0
Age                            0
Gender                         0
Stage                          0
GeneticRisk                    0
TreatmentType                  0
ComorbidityIndex               0
TreatmentResponse              0
SurvivalTime                 160
Censored                       0
GeneticRisk_missing            0
ComorbidityIndex_missing       0
TreatmentResponse_missing      0
dtype: int64


In [54]:
# --- Iterative Imputation with Missing Indicators ---
# Goal: Use multivariate chained regression to infer missing values and keep indicator columns

imputer = IterativeImputer(max_iter=10, random_state=42, sample_posterior=False)

# Important: IterativeImputer does NOT support add_indicator directly → we combine with MissingIndicator
indicator = MissingIndicator()

# Fit imputers
imputer.fit(df[cols_to_impute])
missing_mask = indicator.fit_transform(df[cols_to_impute])

# Perform imputation
transformed = imputer.transform(df[cols_to_impute])

# Build final DataFrame
df_iter_indicator = df.copy()
df_iter_indicator[cols_to_impute] = transformed

# Add indicator columns manually
indicator_cols = [f"{col}_missing" for col in indicator.features_]
df_iter_indicator[indicator_cols] = missing_mask

print("Iterative Imputation + Indicators - Remaining NaNs:\n", df_iter_indicator.isna().sum())

Iterative Imputation + Indicators - Remaining NaNs:
 id                     0
Age                    0
Gender                 0
Stage                  0
GeneticRisk            0
TreatmentType          0
ComorbidityIndex       0
TreatmentResponse      0
SurvivalTime         160
Censored               0
0_missing              0
1_missing              0
2_missing              0
dtype: int64


In [55]:
# --- Missing Indicator Only (Optional) ---
# Goal: Create binary indicator columns for missing values only

indicator = MissingIndicator()
missing_mask = indicator.fit_transform(df[cols_to_impute])

indicator_cols = [f"{col}_missing" for col in indicator.features_]

df_indicator_only = df.copy()
df_indicator_only[indicator_cols] = missing_mask

print("Missing Indicator Only - No Imputation Applied")


Missing Indicator Only - No Imputation Applied
